# HumAID — Zero-shot Classification (Filtered Labels, Batch API, and Auto-Report)

This notebook runs **zero-shot tweet classification** for the HumAID events using ChatGPT, **restricting choices to only the labels that *actually exist* in each event’s ground truth** (“filtered labels”). It submits jobs via the **/v1/chat/completions Batch API**, enforces **token budgeting** across events, and produces per-event **analysis artifacts + an HTML results index**.

---

## What this notebook does

**Pipeline overview**
1. **Discover datasets** under `Dataset/HumAID/<event>/<event>_{train|dev|test}.tsv`.
2. **Token budgeting** (`budget.build_token_index`): estimates total tokens per event (prompt + output).
3. **Key routing**  
   - Small events → `OPENAI_API_KEY_1` (Tier-1).  
   - Large events (over cap) → `OPENAI_API_KEY_2` (Alt key).
4. **Filtered-labels requests** (`batch.build_requests_jsonl_S`)  
   - Extracts **CURRENT_LABELS** = labels present in the event’s ground truth.  
   - Builds a **strict JSON schema** enumerating only those labels.  
   - Prompts the model to choose **exactly one** label from that list.
5. **Single-label bypass** (`runner.run_experiment`)  
   - If an event has **exactly one** valid label, **no API call is made**.  
   - We generate predictions locally (all rows → that label) and still produce full analysis.
6. **Batch submission** (multi-label events): upload JSONL → create batch → wait → download outputs.
7. **Parse + save predictions** and run **evaluation** (`eval.analyze_and_export_mistakes`):
   - Scope = **truth-only** (metrics exclude labels not present in the event).  
   - Records `invalid_pred_outside_truth` for sanity checks.  
   - Exports confusion matrices, per-class metrics, mistakes CSV, and `summary.json`.
8. **Build HTML index** (`report.build_results_index`): interactive, sortable tables, best-run badges.

---

## Why filtered labels?

- Some events miss one or more classes entirely. Allowing the global label set would invite **hallucinated labels** and distort metrics.  
- Restricting to the **ground-truth present labels** aligns the task with real data and keeps metrics meaningful.  
- We still **count** any model outputs outside the event’s label set in `invalid_pred_outside_truth` for QA.

---

## Important settings (edit above as needed)

- **MODEL**: `gpt-4o-mini`  
- **RULES**: `RULES_BASELINE` (see `rules.py`)  
- **TAG**: appended to the run directory for provenance (e.g., `modeS-RULES_BASELINE-filtered`)  
- **DRYRUN_N**: quick sync sanity check before batching  
- **POLL_SECS**: polling interval for batch completion  
- **BATCH_TOKEN_LIMIT**: per-key token budget (e.g., Tier-1 cap)  
- **MAX_OUTPUT_TOKENS**: must match batch builder’s `max_tokens` (default 40)

Environment:
- `.env` must provide `OPENAI_API_KEY_1` and (optionally) `OPENAI_API_KEY_2`.

---

# 0) Setup

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # from budget.py
from humaidclf.batch import use_api_key_env           # context manager for key switching
from rules import RULES_BASELINE

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4o-mini"
RULES = RULES_BASELINE
TAG = "modeS-RULES_BASELINE-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

BATCH_TOKEN_LIMIT = 2_000_000  # Tier-1 cap
SAFETY_MARGIN = 0.90           # 10% headroom
MAX_OUTPUT_TOKENS = 40


# 1) Discover datasets (events/splits)

In [2]:
# --- discover datasets ---
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run with Tier-1 key:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Too big for Tier-1 (use alternate key):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
1,canada_wildfires_2016,test,Dataset\HumAID\canada_wildfires_2016\canada_wi...,445,507,225615,True,11.3
8,kaikoura_earthquake_2016,test,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,435,521,226635,True,11.3
2,cyclone_idai_2019,test,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,779,553,430787,True,21.5
4,hurricane_florence_2018,test,Dataset\HumAID\hurricane_florence_2018\hurrica...,1241,535,663935,True,33.2
7,hurricane_maria_2017,test,Dataset\HumAID\hurricane_maria_2017\hurricane_...,1442,521,751282,True,37.6
0,california_wildfires_2018,test,Dataset\HumAID\california_wildfires_2018\calif...,1461,541,790401,True,39.5
3,hurricane_dorian_2019,test,Dataset\HumAID\hurricane_dorian_2019\hurricane...,1508,537,809796,True,40.5
9,kerala_floods_2018,test,Dataset\HumAID\kerala_floods_2018\kerala_flood...,1582,542,857444,True,42.9
5,hurricane_harvey_2017,test,Dataset\HumAID\hurricane_harvey_2017\hurricane...,1805,520,938600,True,46.9
6,hurricane_irma_2017,test,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,1862,520,968240,True,48.4


OK to run with Tier-1 key:


,event,split,num_rows,est_total_tokens,limit_used_%
0,canada_wildfires_2016,test,445,225615,11.3
1,kaikoura_earthquake_2016,test,435,226635,11.3
2,cyclone_idai_2019,test,779,430787,21.5
3,hurricane_florence_2018,test,1241,663935,33.2
4,hurricane_maria_2017,test,1442,751282,37.6
5,california_wildfires_2018,test,1461,790401,39.5
6,hurricane_dorian_2019,test,1508,809796,40.5
7,kerala_floods_2018,test,1582,857444,42.9
8,hurricane_harvey_2017,test,1805,938600,46.9
9,hurricane_irma_2017,test,1862,968240,48.4


Too big for Tier-1 (use alternate key):


,event,split,num_rows,est_total_tokens,limit_used_%


# 2) Run all datasets (sequentially)

In [3]:
# --- helpers to run a list of datasets ---
def run_list(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)            
            results.append({
                "event": event,
                "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1,
                "accuracy": acc,
                "num_total": n,
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event,
                "split": split,
                "run_dir": "ERROR",
                "predictions_csv": "",
                "macro_f1": float("nan"),
                "accuracy": float("nan"),
                "num_total": 0,
            })
    return pd.DataFrame(results)

# --- 1) Use OPENAI_API_KEY_1 for smaller datasets ---
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using Tier-1 key (OPENAI_API_KEY_1)")
    df_runs_small = run_list(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")
    display(df_runs_small)

# --- 2) Use OPENAI_API_KEY_2 for larger datasets ---
if not df_too_big.empty:
    with use_api_key_env("OPENAI_API_KEY_2"):
        print(">>> Using alternate key (OPENAI_API_KEY_2)")
        df_runs_big = run_list(df_too_big, RULES, MODEL, tag=f"{TAG}-ALT")
        display(df_runs_big)
else:
    df_runs_big = pd.DataFrame()
    print("No large datasets; nothing to run with the alternate key.")

# (optional) save an index of what ran under which key
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

df_runs_small.assign(key="OPENAI_API_KEY_1").to_csv(idx_dir / f"runs_tier1_{MODEL}_{TAG}_{stamp}.csv", index=False)
if not df_runs_big.empty:
    df_runs_big.assign(key="OPENAI_API_KEY_2").to_csv(idx_dir / f"runs_alt_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run indexes in:", idx_dir)

>>> Using Tier-1 key (OPENAI_API_KEY_1)

=== Running canada_wildfires_2016/test (gpt-4o-mini | modeS-RULES_BASELINE-filtered-TIER1) ===
Macro-F1 (tiny sample): 0.8277777777777778
[batch batch_69080dbaaf9c8190a470981105f79822] status = validating
[batch batch_69080dbaaf9c8190a470981105f79822] status = completed
Saved predictions to: runs\canada_wildfires_2016\test\gpt-4o-mini\20251102-180436-modeS-RULES_BASELINE-filtered-TIER1\predictions.csv
Macro-F1: 0.645202109759117

=== Running kaikoura_earthquake_2016/test (gpt-4o-mini | modeS-RULES_BASELINE-filtered-TIER1) ===
Macro-F1 (tiny sample): 0.7062770562770562
[batch batch_69080efe846c8190a1447ed128857e78] status = validating
[batch batch_69080efe846c8190a1447ed128857e78] status = in_progress
[batch batch_69080efe846c8190a1447ed128857e78] status = in_progress
[batch batch_69080efe846c8190a1447ed128857e78] status = in_progress
[batch batch_69080efe846c8190a1447ed128857e78] status = completed
Saved predictions to: runs\kaikoura_earthquake_

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total
0,canada_wildfires_2016,test,runs\canada_wildfires_2016\test\gpt-4o-mini\20...,runs\canada_wildfires_2016\test\gpt-4o-mini\20...,0.645202,0.743820,445
1,kaikoura_earthquake_2016,test,runs\kaikoura_earthquake_2016\test\gpt-4o-mini...,runs\kaikoura_earthquake_2016\test\gpt-4o-mini...,0.685925,0.703448,435
2,cyclone_idai_2019,test,runs\cyclone_idai_2019\test\gpt-4o-mini\202511...,runs\cyclone_idai_2019\test\gpt-4o-mini\202511...,0.582440,0.677792,779
3,hurricane_florence_2018,test,runs\hurricane_florence_2018\test\gpt-4o-mini\...,runs\hurricane_florence_2018\test\gpt-4o-mini\...,0.686273,0.749396,1241
4,hurricane_maria_2017,test,runs\hurricane_maria_2017\test\gpt-4o-mini\202...,runs\hurricane_maria_2017\test\gpt-4o-mini\202...,0.628029,0.653953,1442
5,california_wildfires_2018,test,runs\california_wildfires_2018\test\gpt-4o-min...,runs\california_wildfires_2018\test\gpt-4o-min...,0.607667,0.701574,1461
6,hurricane_dorian_2019,test,runs\hurricane_dorian_2019\test\gpt-4o-mini\20...,runs\hurricane_dorian_2019\test\gpt-4o-mini\20...,0.596174,0.624668,1508
7,kerala_floods_2018,test,runs\kerala_floods_2018\test\gpt-4o-mini\20251...,runs\kerala_floods_2018\test\gpt-4o-mini\20251...,0.542401,0.673198,1582
8,hurricane_harvey_2017,test,runs\hurricane_harvey_2017\test\gpt-4o-mini\20...,runs\hurricane_harvey_2017\test\gpt-4o-mini\20...,0.625421,0.661496,1805
9,hurricane_irma_2017,test,runs\hurricane_irma_2017\test\gpt-4o-mini\2025...,runs\hurricane_irma_2017\test\gpt-4o-mini\2025...,0.608132,0.633727,1862


No large datasets; nothing to run with the alternate key.
Saved run indexes in: runs\_indexes


In [ ]:
from dotenv import load_dotenv; load_dotenv()
from humaidclf.batch import use_api_key_env

from humaidclf import run_experiment
from rules import RULES_1

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment(
        dataset_path="Dataset/HumAID/canada_wildfires_2016/canada_wildfires_2016_test.tsv",
        rules=RULES_1,
        model="gpt-4o-mini",
        tag="modeS-RULES1-filtered",
        dryrun_n=20,
        poll_secs=300,
        do_analysis=True,
    )
summary


In [ ]:
from dotenv import load_dotenv; load_dotenv()
from humaidclf.batch import use_api_key_env

from humaidclf import run_experiment
from rules import RULES_1

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment(
        dataset_path="Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
        rules=RULES_1,
        model="gpt-4o-mini",
        tag="modeS-RULES1-filtered",
        dryrun_n=20,
        poll_secs=300,
        do_analysis=True,
    )
summary
